In [1]:
import os 
import pandas as pd
import re
from pathlib import Path
import numpy as np
from datetime import datetime, timezone
import json

dirs = {
    "nemo_root": r"/Volumes/lab-windingm/data/instruments/behavioural_rigs/plugcamera",
    "all_sheets" :r"/Volumes/lab-windingm/home/shared/behavioural-rigs_data",
    "sensory_sheets" : r"/Volumes/lab-windingm/home/shared/behavioural-rigs_data/sensory-sheets", 
    "mechano_sheets": r"/Volumes/lab-windingm/home/shared/behavioural-rigs_data/mechano-sheets",
    "final-sheets" : r"/Volumes/lab-windingm/home/shared/behavioural-rigs_data/final-weekly-sheets", 
    "backup_sensory": r"/Volumes/lab-windingm/home/shared/behavioural-rigs_data/sensory-backup-sheets",
    "backup_mechano": r"/Volumes/lab-windingm/home/shared/behavioural-rigs_data/mechano-backup-sheets"
}

def read_pupae_counts(experiment_name, nemo_root=dirs["nemo_root"]):
    predictions_dir = Path(nemo_root) / experiment_name / "pupae" / "predictions"

    if not predictions_dir.exists():
        raise FileNotFoundError(f"Predictions folder not found: {predictions_dir}")
    if not predictions_dir.is_dir():
        raise NotADirectoryError(f"Predictions path is not a folder: {predictions_dir}")

    csv_files = sorted(predictions_dir.glob("*.csv"))

    if len(csv_files) == 0:
        raise FileNotFoundError(f"No CSV files found in: {predictions_dir}")

    return pd.read_csv(csv_files[0])

def convert_pupae_date_to_datetime(df_pupae, year):
    df_pupae = df_pupae.copy()
    df_pupae["date"] = df_pupae["date"].astype("string").str.strip()
    date_norm = df_pupae["date"].str.replace("_", "-", regex=False)
    date_full = date_norm + f"-{year}"
    df_pupae["date"] = pd.to_datetime(date_full, format="%d-%m-%Y", errors="coerce")

    return df_pupae

Load pupae batch

In [9]:
year = "2026"

pupae_batch_folder = "VM_2026-02-10_mechano-week-3_batch11_f32" #change accordingly
target_week = 3 #change accordingly

pupae_batch_csv = read_pupae_counts(pupae_batch_folder) #this will only work if you've "connected" to the windingm folder by opening it in your laptop
for d in [pupae_batch_csv]:
    d["pupae_batch_folder"] = pupae_batch_folder

#clean file name
pupae_batch_csv["dataset"] = pupae_batch_csv["dataset"].astype(str).str.strip() #making sure it's clean strings
pupae_batch_csv["dataset"] = pupae_batch_csv["dataset"].str.split("predictions/", n=1).str[-1]
pupae_batch_csv["dataset"] = pupae_batch_csv["dataset"].str.replace(
    r'(\d{1,2})([-_])(\d{1,2})([-_])(day|night)\b',
    r'\1-\2_\4',
    regex=True
)
pattern = r'(?i)(?P<experiment>.+?)[_-](?P<date>\d{1,2}-\d{2})[_-](?P<time>day|night)\_'
pupae_batch_csv[["condition", "date", "time"]] = pupae_batch_csv["dataset"].str.extract(pattern, expand=True)

#extract date of experiment
date_extracted = pupae_batch_csv["date"].str.extract(r'(\d{1,2}[-_]\d{1,2})', expand=False)
date_norm = date_extracted.str.replace("_", "-", regex=False)
pupae_batch_csv["staging_date_clean_dt"] = pd.to_datetime(date_norm + f"-{year}", format="%d-%m-%Y", errors="coerce")

#standardise time of day
pupae_batch_csv["time"] = pupae_batch_csv["time"].str.lower()

#calculate Pupae total
pupae_batch_csv["L_value"] = pupae_batch_csv["dataset"].str.extract(r'(?i)_L(\d+)', expand=False).astype("Int64")
pupae_batch_csv["Pupae"] = pupae_batch_csv["pupae_count"] + pupae_batch_csv["L_value"] #for clarity, the new original sheets should have a column called pupae_final rather than Pupae"

print(
    pupae_batch_csv[
        ["dataset", "condition", "date", "time", "Pupae", "pupae_batch_folder"]
    ].to_string(index=False)
)

Empty DataFrame
Columns: [dataset, condition, date, time, Pupae, pupae_batch_folder]
Index: []


Edit sheets manually, then save them and quit before continuing

In [19]:
mechano_dir = Path(dirs["mechano_sheets"]) 
files = sorted(mechano_dir.glob("*.xlsx"))

dfs = []

for f in files:
    if f.suffix.lower() == ".xlsx":
        tmp = pd.read_excel(f, dtype=str)  # dtype=str avoids Excel auto-typing surprises
    else:
        tmp = pd.read_csv(f, dtype=str)

    tmp.columns = tmp.columns.str.strip()
    tmp["sheet_file"] = f.name
    tmp["sheet_name"] = f.stem
    dfs.append(tmp)

all_sheets = pd.concat(dfs, ignore_index=True)

raw = all_sheets["staging_date"]


staging_str = raw.map(
    lambda x: x.strftime("%d/%m/%Y") if isinstance(x, (pd.Timestamp,)) else ("" if pd.isna(x) else str(x).strip())
)

staging_str = staging_str.replace("", pd.NA)

all_sheets["staging_date"] = staging_str
all_sheets["staging_date_filled"] = all_sheets["staging_date"].ffill()

s = all_sheets["staging_date_filled"].astype("string").str.strip()

all_sheets["time"] = pd.NA
all_sheets.loc[s.str.contains(r"\d\s*-\s*\d", na=False), "time"] = "night"
all_sheets.loc[~s.str.contains(r"\d\s*-\s*\d", na=False), "time"] = "day"

all_sheets["staging_date_clean"] = s.str.replace(
    r"^(?P<d1>\d{1,2})\s*-\s*\d{1,2}\s*/(?P<m>\d{1,2})/(?P<y>\d{4})$",
    r"\g<d1>/\g<m>/\g<y>",
    regex=True
)

all_sheets["staging_date_clean_dt"] = pd.to_datetime(
    all_sheets["staging_date_clean"],
    format="%d/%m/%Y",
    errors="coerce"
)

all_sheets


,experimenter,collector,incubator,shelf,rack,plugcamera,condition,location,staging_date,amendments,...,week,staging_date_filled,staging_date_clean,staging_date_clean_dt,time,pupae_batch_folder,last_edit,Other name,sheet_file,sheet_name
0,LK,HS,1,2,1,73,SS25755,NaN,13-14/01/2026,NaN,...,1,13-14/01/2026,13/01/2026,2026-01-13,night,2026-01-28_LK_mechano-week1_batch3,2026-02-05T13:33:45.876643+00:00,NaN,2026-01-05_mechano-screen_week-1 - Sheet1 (1)....,2026-01-05_mechano-screen_week-1 - Sheet1 (1)
1,LK,HS,1,2,1,74,SS04330,NaN,13-14/01/2026,NaN,...,1,13-14/01/2026,13/01/2026,2026-01-13,night,VM_2026-01-30_LK_mechano-week1-week2_batch5,2026-02-05T13:33:45.876643+00:00,NaN,2026-01-05_mechano-screen_week-1 - Sheet1 (1)....,2026-01-05_mechano-screen_week-1 - Sheet1 (1)
2,LK,HS,1,2,1,75,49826,NaN,13-14/01/2026,-1,...,1,13-14/01/2026,13/01/2026,2026-01-13,night,VM_2026-02-02_mechano-week-1_manual,2026-02-05T13:33:45.876643+00:00,NaN,2026-01-05_mechano-screen_week-1 - Sheet1 (1)....,2026-01-05_mechano-screen_week-1 - Sheet1 (1)
3,LK,HS,1,2,1,76,SS55086,NaN,13-14/01/2026,NaN,...,1,13-14/01/2026,13/01/2026,2026-01-13,night,2026-01-28_LK_mechano-week1_batch3,2026-02-05T13:33:45.876643+00:00,NaN,2026-01-05_mechano-screen_week-1 - Sheet1 (1)....,2026-01-05_mechano-screen_week-1 - Sheet1 (1)
4,LK,HS,1,2,1,96,SS03767,NaN,13-14/01/2026,-1,...,1,13-14/01/2026,13/01/2026,2026-01-13,night,VM_2026-02-02_mechano-week-2_batch6_f14,2026-02-05T13:33:45.876643+00:00,NaN,2026-01-05_mechano-screen_week-1 - Sheet1 (1)....,2026-01-05_mechano-screen_week-1 - Sheet1 (1)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
613,VM,HS,incubator-2,NaN,NaN,209,49826,NaN,2026-01-30 00:00:00,NaN,...,NaN,2026-01-30 00:00:00,2026-01-30 00:00:00,NaT,night,NaN,NaN,NaN,2026-01-19_mechano-screen_week-3 - Sheet1.xlsx,2026-01-19_mechano-screen_week-3 - Sheet1
614,VM,HS,incubator-2,NaN,NaN,210,19-12-GAL4,NaN,2026-01-30 00:00:00,NaN,...,NaN,2026-01-30 00:00:00,2026-01-30 00:00:00,NaT,night,NaN,NaN,NaN,2026-01-19_mechano-screen_week-3 - Sheet1.xlsx,2026-01-19_mechano-screen_week-3 - Sheet1
615,VM,HS,incubator-2,NaN,NaN,211,8769,NaN,2026-01-30 00:00:00,NaN,...,NaN,2026-01-30 00:00:00,2026-01-30 00:00:00,NaT,night,NaN,NaN,NaN,2026-01-19_mechano-screen_week-3 - Sheet1.xlsx,2026-01-19_mechano-screen_week-3 - Sheet1
616,VM,HS,incubator-2,NaN,NaN,212,SS01645,NaN,2026-01-30 00:00:00,NaN,...,NaN,2026-01-30 00:00:00,2026-01-30 00:00:00,NaT,night,NaN,NaN,NaN,2026-01-19_mechano-screen_week-3 - Sheet1.xlsx,2026-01-19_mechano-screen_week-3 - Sheet1



all_sheets["week"] = (
    all_sheets["sheet_file"]
    .str.extract(r"week-(\d+)", expand=False)
    .astype("Int64")
)

if all_sheets["week"].isna().any():
    bad = all_sheets.loc[all_sheets["week"].isna(), "sheet_file"].unique()
    raise ValueError(f"Could not extract week from: {bad[:5]}")

all_sheets["staging_date"] = all_sheets["staging_date"].astype("string").str.strip() #keeps blanks as missing values
all_sheets.loc[all_sheets["staging_date"] == "", "staging_date"] = pd.NA
all_sheets["staging_date_filled"] = all_sheets["staging_date"].ffill() #assuming fill-down for blank rows, where order of the rows matters!

s = all_sheets["staging_date_filled"].astype("string").str.strip()

all_sheets["staging_date_clean"] = s.str.replace(
    r"^(?P<d1>\d{1,2})-\d{1,2}/(?P<m>\d{1,2})/(?P<y>\d{4})$",
    r"\g<d1>/\g<m>/\g<y>",
    regex=True
)
all_sheets["staging_date_clean_dt"] = pd.to_datetime(
    all_sheets["staging_date_clean"],
    dayfirst=True,
    errors="coerce"
)

#assume if there is a dash, assume it is overnight 
all_sheets["time"] = pd.NA
all_sheets.loc[s.str.match(r"^\d{1,2}/\d{1,2}/\d{4}$", na=False), "time"] = "day"
all_sheets.loc[s.str.match(r"^\d{1,2}-\d{1,2}/\d{1,2}/\d{4}$", na=False), "time"] = "night"


In [33]:
all_sheets["staging_date_clean_dt"].value_counts()

staging_date_clean_dt
2026-01-14 00:00:00    86
2026-01-15 00:00:00    86
2026-01-13 00:00:00    43
2026-01-16 00:00:00    43
2026-01-21 00:00:00    42
2026-01-22 00:00:00    42
2026-01-20 00:00:00    21
2026-01-23 00:00:00    21
Name: count, dtype: int64

In [34]:

#create unique experiment ID including week, condition and plug camera
event_key = ["condition", "staging_date_clean_dt", "time"]
plug_key = ["condition", "week", "plugcamera"]

def assert_unique(df, key, label):
    dupes = df[df.duplicated(key, keep=False)]
    if not dupes.empty:
        print(f"\n❌ Duplicate rows for {label}:")
        print(
            dupes[key + ["sheet_file", "sheet_name"]]
            .sort_values(key)
            .head(20)
        )
        raise ValueError(f"Rows are not unique: {label}")

assert_unique(all_sheets, event_key, "event_key")
assert_unique(all_sheets, plug_key, "plugcamera_per_week")

all_sheets["sheet_name"].value_counts()

target_week = 3
# adding batch folder manually (this is temporary until this part is done automatically)
batch = pupae_batch_folder

if "pupae_batch_folder" not in all_sheets.columns:
    all_sheets["pupae_batch_folder"] = pd.NA

p = all_sheets["Pupae"].astype("string").str.strip()
b = all_sheets["pupae_batch_folder"].astype("string").str.strip()

mask = (
    (all_sheets["week"] == target_week) &   # only that week
    p.notna() & (p != "") &                 # Pupae present
    (b.isna() | (b == ""))                  # batch folder missing
)

all_sheets.loc[mask, "pupae_batch_folder"] = batch


#rewriting weekly sheets manually, this will not be done when .csv files naturally merge 
#won't work if the file is open
mechano_dir = Path(dirs["mechano_sheets"])

for sheet_file, df_sheet in all_sheets.groupby("sheet_file"):
    
    if f"week-{target_week}" not in sheet_file:
        continue

    out_path = mechano_dir / sheet_file
    
    cols_to_drop = ["sheet_file", "sheet_name"]
    df_out = df_sheet.drop(columns=[c for c in cols_to_drop if c in df_sheet.columns])

    df_out.to_csv(out_path, index=False)
    print("Overwrote:", out_path)
    
all_sheets["pupae_batch_folder"].value_counts(dropna=False)


❌ Duplicate rows for event_key:
      condition staging_date_clean_dt time  \
419  19-12-GAL4                   NaN  NaN   
458  19-12-GAL4                   NaN  NaN   
497  19-12-GAL4                   NaN  NaN   
536  19-12-GAL4                   NaN  NaN   
575  19-12-GAL4                   NaN  NaN   
614  19-12-GAL4                   NaN  NaN   
398       26259                   NaN  NaN   
437       26259                   NaN  NaN   
476       26259                   NaN  NaN   
515       26259                   NaN  NaN   
554       26259                   NaN  NaN   
593       26259                   NaN  NaN   
402       49766                   NaN  NaN   
441       49766                   NaN  NaN   
480       49766                   NaN  NaN   
519       49766                   NaN  NaN   
558       49766                   NaN  NaN   
597       49766                   NaN  NaN   
418       49826                   NaN  NaN   
457       49826                   NaN  NaN   



ValueError: Rows are not unique: event_key

In [35]:
mechano_dir = Path(dirs["mechano_sheets"]) 
csv_files = sorted(mechano_dir.glob("*.csv"))

dfs = []

for f in csv_files:
    tmp = pd.read_csv(f)
    tmp.columns = tmp.columns.str.strip()   
    tmp["sheet_file"] = f.name             #keeping track of where each row came from
    tmp["sheet_name"] = f.stem              
    dfs.append(tmp)

all_sheets = pd.concat(dfs, ignore_index=True)

all_sheets["week"] = (
    all_sheets["sheet_file"]
    .str.extract(r"week-(\d+)", expand=False)
    .astype("Int64")
)

if all_sheets["week"].isna().any():
    bad = all_sheets.loc[all_sheets["week"].isna(), "sheet_file"].unique()
    raise ValueError(f"Could not extract week from: {bad[:5]}")

all_sheets["staging_date"] = all_sheets["staging_date"].astype("string").str.strip() #keeps blanks as missing values
all_sheets.loc[all_sheets["staging_date"] == "", "staging_date"] = pd.NA
all_sheets["staging_date_filled"] = all_sheets["staging_date"].ffill() #assuming fill-down for blank rows, where order of the rows matters!

s = all_sheets["staging_date_filled"].astype("string").str.strip()

all_sheets["staging_date_clean"] = s.str.replace(
    r"^(?P<d1>\d{1,2})-\d{1,2}/(?P<m>\d{1,2})/(?P<y>\d{4})$",
    r"\g<d1>/\g<m>/\g<y>",
    regex=True
)
all_sheets["staging_date_clean_dt"] = pd.to_datetime(
    all_sheets["staging_date_clean"],
    format="%d/%m/%Y",
    errors="coerce"
)

#assume if there is a dash, assume it is overnight 
all_sheets["time"] = pd.NA
all_sheets.loc[s.str.match(r"^\d{1,2}/\d{1,2}/\d{4}$", na=False), "time"] = "day"
all_sheets.loc[s.str.match(r"^\d{1,2}-\d{1,2}/\d{1,2}/\d{4}$", na=False), "time"] = "night"

#create unique experiment ID including week, condition and plug camera
event_key = ["condition", "staging_date_clean_dt", "time"]
plug_key = ["condition", "week", "plugcamera"]

def assert_unique(df, key, label):
    dupes = df[df.duplicated(key, keep=False)]
    if not dupes.empty:
        print(f"\n❌ Duplicate rows for {label}:")
        print(
            dupes[key + ["sheet_file", "sheet_name"]]
            .sort_values(key)
            .head(20)
        )
        raise ValueError(f"Rows are not unique: {label}")

assert_unique(all_sheets, event_key, "event_key")
assert_unique(all_sheets, plug_key, "plugcamera_per_week")

all_sheets["sheet_name"].value_counts()


ValueError: No objects to concatenate

In [22]:
all_sheets["week"].value_counts()

week
1    258
2    126
Name: count, dtype: int64

In [20]:
pupae_batch_folder = "VM_2026-02-11_mechano-week-3_manual"
batch = pupae_batch_folder
target_week = 3

# ensure column exists
if "pupae_batch_folder" not in all_sheets.columns:
    all_sheets["pupae_batch_folder"] = pd.NA

# robust "Pupae present" check (don’t use string "nan")
pupae_num = pd.to_numeric(all_sheets["Pupae"], errors="coerce")

# robust "batch folder missing" check
bf = all_sheets["pupae_batch_folder"]
bf_str = bf.astype("string").str.strip()

missing_bf = bf.isna() | bf_str.isin(["", "nan", "NaN", "<NA>"])

mask = (all_sheets["week"] == target_week) & pupae_num.notna() & missing_bf
all_sheets.loc[mask, "pupae_batch_folder"] = batch

all_sheets["pupae_batch_folder"].value_counts()

pupae_batch_folder
VM_2026-02-05_mechano-week-2                    84
VM_2026-01-26_mechano-screen_week-1             64
VM_2026-02-02_mechano-week-2_batch6_f14         44
VM_2026-02-02_mechano-week-1_manual             38
VM_2026-01-30_LK_mechano-week1-week2_batch5     30
LK_2026-01-26_mechano-week-1_batch1             30
2026-01-28_LK_mechano-week1_batch3              25
VM_2026-01-05_mechano-week-1-2_batch8_manual    21
VM_2026-01-27_mechano-week-1_batch2_f15         15
VM_2026-01-29_mechano-screen-1_batch4_f10       10
LK_2026-01-27_mechano-week-2_batch2              7
VM_2026-01-30_mechano-screen-1_manual_f6         6
VM_2026-01-28_mechano-week-1_batch3_f5           5
VM_2026-02-05_mechano-week-1-2_batch9_f2         1
VM_2026-02-05_mechano-week-2_manual              1
Name: count, dtype: int64

In [37]:
all_sheets.loc[all_sheets["week"].eq(target_week), "pupae_batch_folder"]\
    .astype("string").str.strip().value_counts(dropna=False).head(20)

bf = all_sheets["pupae_batch_folder"].astype("string").str.strip()
all_sheets.loc[bf.isin(["", "nan", "NaN", "<NA>"]), "pupae_batch_folder"] = pd.NA
all_sheets["pupae_batch_folder"].value_counts()

pupae_batch_folder
VM_2026-02-05_mechano-week-2                    84
VM_2026-01-26_mechano-screen_week-1             64
VM_2026-02-02_mechano-week-2_batch6_f14         44
VM_2026-02-02_mechano-week-1_manual             38
VM_2026-01-30_LK_mechano-week1-week2_batch5     30
LK_2026-01-26_mechano-week-1_batch1             30
2026-01-28_LK_mechano-week1_batch3              25
VM_2026-01-05_mechano-week-1-2_batch8_manual    21
VM_2026-01-27_mechano-week-1_batch2_f15         15
VM_2026-01-29_mechano-screen-1_batch4_f10       10
LK_2026-01-27_mechano-week-2_batch2              7
VM_2026-01-30_mechano-screen-1_manual_f6         6
VM_2026-01-28_mechano-week-1_batch3_f5           5
VM_2026-02-05_mechano-week-1-2_batch9_f2         1
VM_2026-02-05_mechano-week-2_manual              1
Name: count, dtype: int64

In [15]:
#update master sheet (the one we show)
final_dir = Path(dirs["final-sheets"])
snap_dir = final_dir / "snapshots"
final_dir.mkdir(parents=True, exist_ok=True)
snap_dir.mkdir(parents=True, exist_ok=True)

created_at = datetime.now(timezone.utc)
stamp = created_at.strftime("%Y-%m-%dT%H%M%SZ")

all_sheets = all_sheets.copy()
all_sheets["last_edit"] = created_at.isoformat()

latest_path = final_dir / "mechano_sheets_latest.csv"
tmp_latest = final_dir / "mechano_sheets_latest.tmp.csv"

all_sheets.to_csv(tmp_latest, index=False)
tmp_latest.replace(latest_path)

# optional snapshot
#snap_path = snap_dir / f"all_sheets_{stamp}.csv"
#all_sheets.to_csv(snap_path, index=False)

# optional metadata “note”
meta = {
    "last_edit": created_at.isoformat(),
    "latest": latest_path.name,
    #"snapshot": snap_path.name,
    "rows": int(all_sheets.shape[0]),
    "cols": int(all_sheets.shape[1]),
    "inputs_dir": str(dirs["mechano_sheets"]),
}
(meta_path := snap_dir / f"mechano_sheets_{stamp}.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

print("Wrote latest:", latest_path)
#print("Wrote snapshot:", snap_path)
print("Wrote meta:", meta_path)


Wrote latest: /Volumes/lab-windingm/home/shared/behavioural-rigs_data/final-weekly-sheets/mechano_sheets_latest.csv
Wrote meta: /Volumes/lab-windingm/home/shared/behavioural-rigs_data/final-weekly-sheets/snapshots/mechano_sheets_2026-02-10T164039Z.json


Attempting to merge automatically

In [18]:
#create unique experiment ID including week, condition and plug camera
event_key = ["condition", "staging_date_clean_dt", "time"]
plug_key = ["condition", "week", "plugcamera"]

def assert_unique(df, key, label):
    dupes = df[df.duplicated(key, keep=False)]
    if not dupes.empty:
        print(f"\n❌ Duplicate rows for {label}:")
        print(
            dupes[key + ["sheet_file", "sheet_name"]]
            .sort_values(key)
            .head(20)
        )
        raise ValueError(f"Rows are not unique: {label}")
    
assert_unique(pupae_batch_csv, event_key, "event_key")

Not checked from here

In [ ]:
#check for duplicates in both dfs
for d in [all_sheets, pupae_batch_csv]:
    d["condition"] = d["condition"].astype("string").str.strip().str.lower()
    d["time"] = d["time"].astype("string").str.strip().str.lower()
    d["staging_date_clean_dt"] = pd.to_datetime(d["staging_date_clean_dt"], errors="coerce").dt.normalize()

pupae_batch_duplicates = pupae_batch_csv[pupae_batch_csv.duplicated(key, keep= False)].sort_values(key)
pupae_batch_clean = pupae_batch_csv.drop(index=pupae_batch_duplicates.index).copy()

pupae_batch_flagged = pupae_batch_clean.merge(
    dup_keys.assign(in_sheet_duplicates=True),
    on=key,
    how="left"
)
pupae_batch_flagged["in_sheet_duplicates"] = pupae_batch_flagged["in_sheet_duplicates"].fillna(False)

#exclude pupae batch rows giving issues
bad_rows_pupae_batch = pupae_batch_flagged[pupae_batch_flagged["condition"].isna() | pupae_batch_flagged["date"].isna() | pupae_batch_flagged["time"].isna()]
pupae_batch_drop_count_dups = pupae_batch_flagged.drop(index=bad_rows_pupae_batch.index).copy()

pupae_batch_final = pupae_batch_drop_count_dups[~pupae_batch_drop_count_dups["in_sheet_duplicates"]].copy()
manual_add_dups = pupae_batch_drop_count_dups[pupae_batch_drop_count_dups["in_sheet_duplicates"]].copy()

pupae_in = pupae_batch_final.loc[:, key + ["Pupae", "pupae_batch_folder"]].rename(
    columns={"Pupae": "_pupae_in", "pupae_batch_folder": "_pupae_folder_in"}
)

manually_add_counts_df = pd.concat([bad_rows_pupae_batch, pupae_batch_duplicates, manual_add_dups])

out_dir = Path(dirs["main"])  
out_dir.mkdir(parents=True, exist_ok=True)

#manually_add_counts_df.to_csv(
    #out_dir / f"dups_add_manually_{pupae_batch_folder}.csv",
    #index=False
#)

#manually_add_counts_df.to_csv(f"manually_add_{pupae_batch_folder}.csv")

pupae_in_tidy = pupae_in.sort_values(by="staging_date_clean_dt", ascending=True)
#pupae_in_tidy.to_csv(
    #out_dir / f"pupae_in_tidy_{pupae_batch_folder}.csv", 
    #index=False
#)

pupae_in_tidy.to_csv(f"pupae_in_tidy_{pupae_batch_folder}.csv")

if not manually_add_counts_df.empty:
    raise ValueError(
        f"{len(manually_add_counts_df)} rows need manual review (manually_add_counts_df not empty).\n"
        f"File written to: {out_dir}"
    )

pupae_in_tidy = pupae_in.sort_values(by="staging_date_clean_dt", ascending=True)
#pupae_in_tidy.to_csv(
    #out_dir / f"pupae_in_tidy_{pupae_batch_folder}.csv", 
    #index=False
#)

#pupae_in_tidy.to_csv(f"pupae_in_tidy_{pupae_batch_folder}.csv")

Load weekly sheets

In [ ]:
all_sheets = Path(dirs["mechano_sheets"]) 
csv_files = sorted(all_sheets.glob("*.csv"))

dfs = []

for f in csv_files:
    tmp = pd.read_csv(f)
    tmp.columns = tmp.columns.str.strip()   
    tmp["sheet_file"] = f.name             #keeping track of where each row came from
    tmp["sheet_name"] = f.stem              
    dfs.append(tmp)

all_sheets = pd.concat(dfs, ignore_index=True)

all_sheets["week"] = (
    all_sheets["sheet_file"]
    .str.extract(r"week-(\d+)", expand=False)
    .astype("Int64")
)

if all_sheets["week"].isna().any():
    bad = all_sheets.loc[all_sheets["week"].isna(), "sheet_file"].unique()
    raise ValueError(f"Could not extract week from: {bad[:5]}")

all_sheets["staging_date"] = all_sheets["staging_date"].astype("string").str.strip() #keeps blanks as missing values
all_sheets.loc[all_sheets["staging_date"] == "", "staging_date"] = pd.NA
all_sheets["staging_date_filled"] = all_sheets["staging_date"].ffill() #assuming fill-down for blank rows, where order of the rows matters!

s = all_sheets["staging_date_filled"].astype("string").str.strip()

all_sheets["staging_date_clean"] = s.str.replace(
    r"^(?P<d1>\d{1,2})-\d{1,2}/(?P<m>\d{1,2})/(?P<y>\d{4})$",
    r"\g<d1>/\g<m>/\g<y>",
    regex=True
)
all_sheets["staging_date_clean_dt"] = pd.to_datetime(
    all_sheets["staging_date_clean"],
    format="%d/%m/%Y",
    errors="coerce"
)

#assume if there is a dash, assume it is overnight 
all_sheets["time"] = pd.NA
all_sheets.loc[s.str.match(r"^\d{1,2}/\d{1,2}/\d{4}$", na=False), "time"] = "day"
all_sheets.loc[s.str.match(r"^\d{1,2}-\d{1,2}/\d{1,2}/\d{4}$", na=False), "time"] = "night"

#create unique experiment ID including week, condition and plug camera
event_key = ["condition", "staging_date_clean_dt", "time"]
plug_key = ["condition", "week", "plugcamera"]

def assert_unique(df, key, label):
    dupes = df[df.duplicated(key, keep=False)]
    if not dupes.empty:
        print(f"\n❌ Duplicate rows for {label}:")
        print(
            dupes[key + ["sheet_file", "sheet_name"]]
            .sort_values(key)
            .head(20)
        )
        raise ValueError(f"Rows are not unique: {label}")

assert_unique(all_sheets, event_key, "event_key")
assert_unique(all_sheets, plug_key, "plugcamera_per_week")

In [ ]:
def safe_overwrite_csv(df: pd.DataFrame, target_path: Path) -> None:
    target_path.parent.mkdir(parents=True, exist_ok=True)

    tmp_path = target_path.with_suffix(target_path.suffix + ".tmp")

    # If you don't want to write the helper columns back out, drop them here:
    out = df.drop(columns=["sheet_file", "sheet_name"], errors="ignore")

    out.to_csv(tmp_path, index=False)
    tmp_path.replace(target_path)  # atomic on same filesystem


In [7]:
completed_sheet = pd.read_csv("260113_all_sheets.csv")
completed_sheet_missing = completed_sheet[completed_sheet["pupae"] == "n/k"]
#completed_sheet_missing.to_csv("250112_pupae_missing.csv")
completed_sheet_missing

,Unnamed: 0.1,experimenter,collector,shelf,rack,plugcamera,condition,location,staging_date,amendments,...,tues_am_removed_eggs_binary,time,incubator,mon_removed_eggs_binary,tues_removed_pm_eggs_binary,weds_am_mortality_n,fri_pm_removed_eggs_binary,staging_date_filled,staging_date_clean,staging_date_clean_dt
486,486.0,NaN,NaN,NaN,NaN,51,control-1,NaN,6-7/11/2025,NaN,...,NaN,night,NaN,NaN,NaN,NaN,NaN,6-7/11/2025,06/11/2025,06/11/2025
492,492.0,NaN,NaN,NaN,NaN,58,SS01705,T1/A1,6-7/11/2025,NaN,...,NaN,night,NaN,NaN,NaN,NaN,NaN,6-7/11/2025,06/11/2025,06/11/2025
493,493.0,NaN,NaN,NaN,NaN,59,SS01716,T1/A4,6-7/11/2025,NaN,...,NaN,night,NaN,NaN,NaN,NaN,NaN,6-7/11/2025,06/11/2025,06/11/2025
502,502.0,NaN,NaN,NaN,NaN,195,SS01973,T1/A5,6-7/11/2025,NaN,...,NaN,night,NaN,NaN,NaN,NaN,NaN,6-7/11/2025,06/11/2025,06/11/2025
582,582.0,NaN,NaN,NaN,NaN,27,MB143b,T1/C9,12-13/11/2025,NaN,...,NaN,night,NaN,NaN,NaN,NaN,NaN,12-13/11/2025,12/11/2025,12/11/2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1682,1681.0,LK,HS,1.0,2.0,32,control-1,NaN,11/12/2025,NaN,...,NaN,day,1,NaN,NaN,NaN,NaN,11/12/2025,11/12/2025,11/12/2025
1694,1693.0,LK,HS,1.0,2.0,44,MB145B,NaN,11/12/2025,NaN,...,NaN,day,1,NaN,NaN,NaN,NaN,11/12/2025,11/12/2025,11/12/2025
1699,1698.0,LK,HS,1.0,2.0,169,control-2,NaN,11/12/2025,NaN,...,NaN,day,1,NaN,NaN,NaN,NaN,11/12/2025,11/12/2025,11/12/2025
1701,1700.0,LK,HS,1.0,2.0,171,MB328b,NaN,11/12/2025,NaN,...,NaN,day,1,NaN,NaN,NaN,NaN,11/12/2025,11/12/2025,11/12/2025


In [4]:
key = ["staging_date_clean_dt", "time", "condition"]
sheet_duplicates = all_sheets[all_sheets.duplicated(key, keep=False)].sort_values(key)
dup_keys = sheet_duplicates[key].drop_duplicates()

In [5]:
sheet_flagged = all_sheets.merge(
    dup_keys.assign(in_sheet_duplicates=True), 
    on=key,
    how="left"
)

sheet_flagged["in_sheet_duplicates"] = sheet_flagged["in_sheet_duplicates"].fillna(False)
#sheet_clean = sheet_flagged.drop(index=sheet_duplicates.index).copy() #makes rows disappear

/var/folders/vp/kct0bnpj5g9_h6x43mk9x6n00000gp/T/ipykernel_1487/929419192.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sheet_flagged["in_sheet_duplicates"] = sheet_flagged["in_sheet_duplicates"].fillna(False)


Load pupae counts batch

In [6]:
pupae_batch_folder = "VM_2025-12-09-screen_f65_retry" #change accordingly
pupae_batch_csv = read_pupae_counts(pupae_batch_folder) #this will only work if you've "connected" to the windingm folder by opening it in your laptop
for d in [pupae_batch_csv]:
    d["pupae_batch_folder"] = pupae_batch_folder

#clean file name
pupae_batch_csv["dataset"] = pupae_batch_csv["dataset"].astype(str).str.strip() #making sure it's clean strings
pupae_batch_csv["dataset"] = pupae_batch_csv["dataset"].str.split("predictions/", n=1).str[-1]
pupae_batch_csv["dataset"] = pupae_batch_csv["dataset"].str.replace(
    r'(\d{1,2})([-_])(\d{1,2})([-_])(day|night)\b',
    r'\1-\2_\4',
    regex=True
)
pattern = r'(?P<experiment>.+?)[_-](?P<date>\d{1,2}-\d{2})[_-](?P<time>day|night)\_'
pupae_batch_csv[["condition", "date", "time"]] = pupae_batch_csv["dataset"].str.extract(pattern, expand=True)

#extract date of experiment
year = "2025"
date_extracted = pupae_batch_csv["date"].str.extract(r'(\d{1,2}[-_]\d{1,2})', expand=False)
date_norm = date_extracted.str.replace("_", "-", regex=False)
pupae_batch_csv["staging_date_clean_dt"] = pd.to_datetime(date_norm + f"-{year}", format="%d-%m-%Y", errors="coerce")

#standardise time of day
pupae_batch_csv["time"] = pupae_batch_csv["time"].str.lower()

#calculate Pupae total
pupae_batch_csv["L_value"] = pupae_batch_csv["dataset"].str.extract(r'_L(\d+)', expand=False).astype("Int64")
pupae_batch_csv["Pupae"] = pupae_batch_csv["pupae_count"] + pupae_batch_csv["L_value"] #for clarity, the new original sheets should have a column called pupae_final rather than Pupae"

#check for duplicates in both dfs
for d in [all_sheets, pupae_batch_csv]:
    d["condition"] = d["condition"].astype("string").str.strip().str.lower()
    d["time"] = d["time"].astype("string").str.strip().str.lower()
    d["staging_date_clean_dt"] = pd.to_datetime(d["staging_date_clean_dt"], errors="coerce").dt.normalize()

pupae_batch_duplicates = pupae_batch_csv[pupae_batch_csv.duplicated(key, keep= False)].sort_values(key)
pupae_batch_clean = pupae_batch_csv.drop(index=pupae_batch_duplicates.index).copy()

pupae_batch_flagged = pupae_batch_clean.merge(
    dup_keys.assign(in_sheet_duplicates=True),
    on=key,
    how="left"
)
pupae_batch_flagged["in_sheet_duplicates"] = pupae_batch_flagged["in_sheet_duplicates"].fillna(False)

#exclude pupae batch rows giving issues
bad_rows_pupae_batch = pupae_batch_flagged[pupae_batch_flagged["condition"].isna() | pupae_batch_flagged["date"].isna() | pupae_batch_flagged["time"].isna()]
pupae_batch_drop_count_dups = pupae_batch_flagged.drop(index=bad_rows_pupae_batch.index).copy()

pupae_batch_final = pupae_batch_drop_count_dups[~pupae_batch_drop_count_dups["in_sheet_duplicates"]].copy()
manual_add_dups = pupae_batch_drop_count_dups[pupae_batch_drop_count_dups["in_sheet_duplicates"]].copy()

pupae_in = pupae_batch_final.loc[:, key + ["Pupae", "pupae_batch_folder"]].rename(
    columns={"Pupae": "_pupae_in", "pupae_batch_folder": "_pupae_folder_in"}
)

manually_add_counts_df = pd.concat([bad_rows_pupae_batch, pupae_batch_duplicates, manual_add_dups])

out_dir = Path(dirs["main"])  
out_dir.mkdir(parents=True, exist_ok=True)

#manually_add_counts_df.to_csv(
    #out_dir / f"dups_add_manually_{pupae_batch_folder}.csv",
    #index=False
#)

manually_add_counts_df.to_csv(f"manually_add_{pupae_batch_folder}.csv")

pupae_in_tidy = pupae_in.sort_values(by="staging_date_clean_dt", ascending=True)
#pupae_in_tidy.to_csv(
    #out_dir / f"pupae_in_tidy_{pupae_batch_folder}.csv", 
    #index=False
#)

pupae_in_tidy.to_csv(f"pupae_in_tidy_{pupae_batch_folder}.csv")

if not manually_add_counts_df.empty:
    raise ValueError(
        f"{len(manually_add_counts_df)} rows need manual review (manually_add_counts_df not empty).\n"
        f"File written to: {out_dir}"
    )

pupae_in_tidy = pupae_in.sort_values(by="staging_date_clean_dt", ascending=True)
#pupae_in_tidy.to_csv(
    #out_dir / f"pupae_in_tidy_{pupae_batch_folder}.csv", 
    #index=False
#)

pupae_in_tidy.to_csv(f"pupae_in_tidy_{pupae_batch_folder}.csv")

/var/folders/vp/kct0bnpj5g9_h6x43mk9x6n00000gp/T/ipykernel_1487/1062507532.py:44: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pupae_batch_flagged["in_sheet_duplicates"] = pupae_batch_flagged["in_sheet_duplicates"].fillna(False)
/var/folders/vp/kct0bnpj5g9_h6x43mk9x6n00000gp/T/ipykernel_1487/1062507532.py:57: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  manually_add_counts_df = pd.concat([bad_rows_pupae_batch, pupae_batch_duplicates, manual_add_dups])


ValueError: 2 rows need manual review (manually_add_counts_df not empty).
File written to: /Volumes/lab-windingm/home/shared/behavioural-rigs_data

In [36]:
pupae_batch_csv

,pupae_count,dataset,pupae_batch_folder,condition,date,time,staging_date_clean_dt,L_value,Pupae
0,171,SS01972_27-11_night_L40.mp4.json,LK_2025-12-11_split-week18-pupae,ss01972,27-11,night,2025-11-27,40,211
1,161,SS01776_28-11_day_L10.mp4.json,LK_2025-12-11_split-week18-pupae,ss01776,28-11,day,2025-11-28,10,171
2,158,MB328B_27-11_day_L27.mp4.json,LK_2025-12-11_split-week18-pupae,mb328b,27-11,day,2025-11-27,27,185
3,168,SS01777_27-11_day_L24.mp4.json,LK_2025-12-11_split-week18-pupae,ss01777,27-11,day,2025-11-27,24,192
4,166,MB065B_27-11_night_L18.mp4.json,LK_2025-12-11_split-week18-pupae,mb065b,27-11,night,2025-11-27,18,184
5,164,SS04559_27-11_day_L13.mp4.json,LK_2025-12-11_split-week18-pupae,ss04559,27-11,day,2025-11-27,13,177
6,136,SS01716_27-11_night_L14.mp4.json,LK_2025-12-11_split-week18-pupae,ss01716,27-11,night,2025-11-27,14,150
7,158,SS01777_28-11_day_L5.mp4.json,LK_2025-12-11_split-week18-pupae,ss01777,28-11,day,2025-11-28,5,163
8,132,SS02181_27-11_night_L34.mp4.json,LK_2025-12-11_split-week18-pupae,ss02181,27-11,night,2025-11-27,34,166
9,186,SS02163_27-11_day_L24.mp4.json,LK_2025-12-11_split-week18-pupae,ss02163,27-11,day,2025-11-27,24,210


Delete

In [ ]:
#preparing for merging, build pupae table with unique keys
merged_df = all_sheets.merge(
    pupae_in,
    on=key, 
    how='left', 
    validate="1:1"
)

target = "Pupae_final"

if target not in merged_df.columns:
    merged_df[target] = pd.NA
if "pupae_batch_folder" not in merged_df.columns:
    merged_df["pupae_batch_folder"] = pd.NA

merged_df[target] = merged_df[target].fillna(merged_df["_pupae_in"])

merged_df[target] = merged_df[target].fillna(merged_df["_pupae_in"])
merged_df["pupae_batch_folder"] = merged_df["pupae_batch_folder"].fillna(merged_df["_pupae_folder_in"])

target_col = "pupae_final"
# fill only missing
merged_df[target_col] = merged_df[target_col].fillna(merged_df["Pupae_new"])
merged_df = merged_df.drop(columns=["Pupae_new"], errors="ignore")

cols = ['pupae_total','pupae', 'Pupae_x', 'Pupae_y', "Pupae"]
populating_pupae_counter = merged_df[["experimenter", "plugcamera", "condition", "amendments", "comments",
                                      "Pupae_x", "Pupae_y", "pupae", "Pupae", "staging_date", "staging_times", "staging_date_clean_dt", "time",
                                      "sheet_name", "sheet_file", "pupae_batch_folder" ]].copy()

populating_pupae_counter[cols] = populating_pupae_counter[cols].replace('', np.nan)

pupae_cols = [c for c in populating_pupae_counter.columns if "pupae" in c.lower()]
assert pupae_cols == ["pupae_final", "pupae_batch_folder"] or set(pupae_cols) == {"pupae_final", "pupae_batch_folder"}, pupae_cols

MergeError: Merge keys are not unique in left dataset; not a one-to-one merge

In [35]:
populating_pupae_counter

,experimenter,plugcamera,condition,amendments,comments,Pupae_x,Pupae_y,pupae,Pupae,staging_date,staging_times,staging_date_clean_dt,time,sheet_name,sheet_file,pupae_batch_folder
0,NaN,145.0,control-1,-1.0,NaN,NaN,NaN,NaN,NaN,14-15/10/2025,16:45-9:00,2025-10-14,night,2025-10-06_split-gal4-week-12 - Sheet1 (1),2025-10-06_split-gal4-week-12 - Sheet1 (1).csv,NaN
1,NaN,146.0,control-2,NaN,recordings started at 9:45,NaN,NaN,NaN,NaN,14-15/10/2025,16:45-9:00,2025-10-14,night,2025-10-06_split-gal4-week-12 - Sheet1 (1),2025-10-06_split-gal4-week-12 - Sheet1 (1).csv,NaN
2,NaN,147.0,ss01989,NaN,NaN,NaN,NaN,NaN,NaN,14-15/10/2025,16:45-9:00,2025-10-14,night,2025-10-06_split-gal4-week-12 - Sheet1 (1),2025-10-06_split-gal4-week-12 - Sheet1 (1).csv,NaN
3,NaN,148.0,t13a06,NaN,NaN,NaN,NaN,NaN,NaN,14-15/10/2025,16:45-9:00,2025-10-14,night,2025-10-06_split-gal4-week-12 - Sheet1 (1),2025-10-06_split-gal4-week-12 - Sheet1 (1).csv,NaN
4,NaN,149.0,t17a8,NaN,"not 149, actually recorded with plugcamera 150",NaN,NaN,NaN,NaN,14-15/10/2025,16:45-9:00,2025-10-14,night,2025-10-06_split-gal4-week-12 - Sheet1 (1),2025-10-06_split-gal4-week-12 - Sheet1 (1).csv,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1770,LK,NaN,ss01745,NaN,Not recorded,NaN,NaN,NaN,NaN,12/12/2025,NaN,2025-12-12,day,2025-12-01_split-gal4-week-20 - Sheet1,2025-12-01_split-gal4-week-20 - Sheet1.csv,NaN
1771,LK,NaN,ss01417,NaN,Not recorded,NaN,NaN,NaN,NaN,12/12/2025,NaN,2025-12-12,day,2025-12-01_split-gal4-week-20 - Sheet1,2025-12-01_split-gal4-week-20 - Sheet1.csv,NaN
1772,LK,NaN,ss01757,NaN,Not recorded,NaN,NaN,NaN,NaN,12/12/2025,NaN,2025-12-12,day,2025-12-01_split-gal4-week-20 - Sheet1,2025-12-01_split-gal4-week-20 - Sheet1.csv,NaN
1773,LK,NaN,ss00864,NaN,Not recorded,NaN,NaN,NaN,NaN,12/12/2025,NaN,2025-12-12,day,2025-12-01_split-gal4-week-20 - Sheet1,2025-12-01_split-gal4-week-20 - Sheet1.csv,NaN


KeyError: "['pupae_total'] not in index"

In [32]:
weekly_dir = Path(dirs["weekly-sheets"])

for sheet_file, grp in populating_pupae_counter.groupby("sheet_file", sort=False):
    safe_overwrite_csv(grp, weekly_dir / sheet_file)

In [31]:
#do checks on the relevant files
which_sheets = merged_df[merged_df["pupae_batch_folder"].notna()]
which_sheets["sheet_file"].value_counts()

sheet_file
2025-11-10_split-gal4-week-17 - Sheet1.csv    98
2025-11-24_split-gal4-week-19 - Sheet1.csv    67
2025-10-27_split-week-15 - Sheet1.csv         36
2025-11-03_split-gal4-week-16 - Sheet1.csv    33
Name: count, dtype: int64

In [ ]:
merged_keys = merged_df.loc[merged_df["pupae_batch_folder"].notna(), key].drop_duplicates()

pupae_keys = pupae_batch_drop_count_dups[key].drop_duplicates()

missing_keys = pupae_keys.merge(merged_keys, on=key, how="left", indicator=True)
missing_keys = missing_keys[missing_keys["_merge"] == "left_only"].drop(columns="_merge")

missing_pupae_rows = pupae_batch_drop_count_dups.merge(missing_keys, on=key, how="inner")

print("Pupae rows that did not end up in merged_df:")
print(missing_pupae_rows[key + ["dataset", "pupae_count", "L_value", "Pupae", "pupae_batch_folder"]]
      .to_string(index=False))

In [ ]:
out_dir = Path(dirs["main"])  
out_dir.mkdir(parents=True, exist_ok=True)

missing_pupae_rows.to_csv(
    out_dir / f"add_manually_{pupae_batch_folder}.csv",
    index=False
)